In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.\
            builder.\
                appName("String Transformation").\
                    getOrCreate()

In [0]:
emp_schema = "id INT, name STRING, department STRING, salary DOUBLE, hire_date STRING, country STRING, age INT, gender STRING"

emp_data = [
    (1, "John Smith", "Engineering", 85000.0, "2021-03-15", "USA", 30, "Male"),
    (2, "Alice Johnson", "Engineering", 92000.0, "2020-07-01", "USA", 35, "Female"),
    (3, "Bob Williams", "Sales", 65000.0, "2022-01-20", "UK", 28, None),
    (4, "Carol Brown", "Sales", 68000.0, "2019-11-05", "UK", 40, "Female"),
    (5, "David Jones", "HR", 55000.0, "2021-09-10", "Canada", 32, "Male"),
    (6, "Eva Garcia", "HR", 58000.0, "2023-02-28", "Canada", 26, "Female"),
    (7, "Frank Miller", "Engineering", 78000.0, "2018-05-22", "Germany", 45, "Male"),
    (8, "Grace Davis", "Marketing", 72000.0, "2022-08-14", "Germany", 29, "Female"),
    (9, "Henry Wilson", "Marketing", 70000.0, "2020-12-01", "India", 33, "Other"),
    (10, "Ivy Taylor", "Finance", 95000.0, "2017-06-30", "India", 50, "Female"),
    (11, "Jack Anderson", "Finance", 89000.0, "2021-04-18", "Australia", 38, "Male"),
    (12, "Karen Thomas", "Engineering", 81000.0, "2023-03-07", "Australia", 25, "Female"),
    (13, "Leo Moore", "Sales", 62000.0, "2019-10-25", "USA", 37, None),
    (14, "Mia Martin", "Engineering", 90000.0, "2022-05-11", "USA", 27, "Female"),
    (15, "Nick Lee", "HR", 60000.0, "2018-02-09", "UK", 42, "Male"),
    (16, "Olivia Harris", "Marketing", 74000.0, "2020-09-03", "Canada", 31, "Female"),
    (17, "Paul Clark", "Finance", 87000.0, "2023-07-19", "Germany", 24, "Male"),
    (18, "Quinn Lewis", "Engineering", 83000.0, "2021-12-12", "India", 34, "Other"),
    (19, "Rachel Walker", "Sales", 66000.0, "2019-08-08", "Australia", 36, "Female"),
    (20, "Sam Hall", "Marketing", 71000.0, "2022-11-27", "USA", 23, None),
]

## Creating the dataFrame
emp = spark.createDataFrame(
            data = emp_data,
            schema = emp_schema
)

In [0]:
emp.show(5)
emp.printSchema()

In [0]:
# Select emp_id, name, salary, gender
# "M" for male "F" for female and null for other type
from pyspark.sql.functions import when, col

emp_gender_filter = emp.withColumn('new_gender', when(col("gender") == "Male", "M")
                                                 .when(col("gender") == "Female", "F")
                                                 .otherwise(None))

emp_gender_filter.show()

In [0]:
# Select emp_id, name, salary, gender
# "M" for male "F" for female and null for other type
from pyspark.sql.functions import expr

emp_gender_filter_expr = emp.withColumn('new_gender', 
                                            expr(
                                                "CASE \
                                                    WHEN gender = 'Male' THEN 'M' \
                                                    WHEN gender = 'Female' THEN 'F' \
                                                    ELSE NULL \
                                                END")
                                        )

emp_gender_filter_expr.show()

In [0]:
## regex_replace
# replace 'i' wiht "I"

from pyspark.sql.functions import regexp_replace

i_with_II = emp_gender_filter.\
                    withColumn("new_name", regexp_replace(col('name'), 'i', 'I'))

i_with_II.show()

In [0]:
# fixing the 'hire_data' dataType
from pyspark.sql.functions import to_date

# with method `withColumn` we can create or replace the Column
emp_date_fix = i_with_II.\
                        withColumn(
                                'hire_date', to_date(col('hire_date'), 'yyyy-MM-dd'))
                                
print(emp.schema['hire_date'])
print(emp_date_fix.schema['hire_date'])

In [0]:
# add current date and current timestamp
from pyspark.sql.functions import current_date, current_timestamp

emp_time_date = emp_date_fix.withColumn('current_date', current_date()).\
                             withColumn('current_timestamp', current_timestamp())

emp_time_date.show(truncate=False)

In [0]:
# Drop Null gender
emp_clean = emp_time_date.na.drop()
emp_clean.show()

#### COALESCE in PySpark

**PySpark also supports coalesce() for handling NULL values.**

* There are two commonly used ways to use it:
    - coalesce() from pyspark.sql.functions — returns the first non-NULL column value.
    - DataFrame.coalesce() — reduces the number of partitions.


In [0]:
# fixing null values to others i.e. 'O'
from pyspark.sql.functions import lit, coalesce

emp_gender_fill = emp_time_date.\
                                withColumn('new_gender', coalesce(col('new_gender'), lit('O')))
                                           
emp_gender_fill.show()                                           

In [0]:
# drop some columns and rename their column name
emp_final = emp_gender_fill.drop('name', 'gender').\
                                                withColumnRenamed('new_name', 'name').\
                                                    withColumnRenamed('new_gender', 'gender')

emp_final.show()                                                    

In [0]:
# convert date intro string format
from pyspark.sql.functions import date_format

date_string = emp_final.withColumn('date_string', date_format(col('hire_date'), 'dd/MM/yyyy'))
date_string.show(5)
print(date_string.schema['date_string'])

In [0]:
# extracting the info from datestring
info_year = date_string.withColumn('year', date_format(col("hire_date"), 'yyyy'))
info_timezone = date_string.withColumn('time_zone', date_format(col('current_timestamp'), 'z'))
info_hour = date_string.withColumn('Hour', date_format(col('current_timestamp'), 'H'))

info_year.show(5, truncate=False)
info_timezone.show(5, truncate=False)
info_hour.show(5, truncate=False)